In [1]:
#I am running this mostly because it was giving me errors
#%pip uninstall streamlit -y
#run this if you need to instale gradio still otherwise keep it like this
#%pip install --upgrade gradio
#%pip install --upgrade Pillow
#This was for error in the next one
#%pip install ipywidgets

In [2]:
#Put here all the imports needed
import gradio as gr
from PIL import Image, ImageDraw
import os
from os import listdir
from pathlib import Path
import numpy as np
import random
#This will be where we then put in the images. Create a new block under this with the two sets and stuff to keep track of 
#which ones we have had and which ones we did not
home = Path.home()
folder_img = home / "OneDrive" / "Documenten" / "PKG - CDD-CESM" / "CDD-CESM" / "Subtracted images of CDD-CESM"

In [3]:
#So this is for looping throug images in folder_img
images_path = []
for images in os.listdir(folder_img):
    #check if image
    if(images.lower().endswith(".jpg")):
        images_path.append(folder_img/images)

#Also keep track of which images we have had
clicked = set()

In [6]:
def load_image ():
    """
    This will simily load the image so the person can click on it
    """
    #First check if list is not empty
    if not images_path:
        raise ValueError("No more images to load!")
    #Pop from the list with images and add it to the set clicked
    path = images_path.pop(random.randrange(0,len(images_path)))
    clicked.add(path)
    #Open the image. usefull doc https://pillow.readthedocs.io/en/stable/reference/Image.html 
    img = Image.open(path)

    return np.array(img)

#Dict met coords van meerdere punten
end_coords = []
def draw_circle (image_input, evt: gr.SelectData):
    """
    Okay so this code will draw a circle around teh clicked pixel and returns the image and coordinates.
    """
    #First getting the coords
    x_coords = evt.index[0]
    y_coords = evt.index[1]

    #Then we open the image with PIL
    img = Image.fromarray(image_input)

    #Create the radius for the circle around the clicked pixel
    radius = 50

    # Draw the cicle around the click location
    #This is from PIL and the usefull docs are https://pillow.readthedocs.io/en/stable/reference/ImageDraw.html
    draw = ImageDraw.Draw(img)
    draw.ellipse(
        #middel of the circle and the line of the circle? I think? is my math mathing?
        [(x_coords - radius, y_coords - radius), (x_coords + radius, y_coords + radius)],
        #This is just the colour of the circle and we do not want fill. I chose standard red but this can be edited
        outline = "red",
        #width of the outline
        width =  5
    )
    end_coords.append((x_coords, y_coords))
    #And then the output of the coords itself but that is what I think we are going to compare in with the NN
    coords = f"{end_coords}"
    
    
    return np.array(img), coords
    
#This is too select the image to work with
first_image = load_image()

#Create the demo
with gr.Blocks() as demo:
    #Add the instructions
    gr.Markdown("Click on the image to select the tumor")

    #Save original image in a state
    with gr.Row():
        #Create a variable with the image, value is the np array, type is thus numpy and it is intractive.
        scan_canvas = gr.Image(value=first_image, type="numpy", interactive=True)
        
        with gr.Column():
            #cordes output + image output
            coords_output = gr.Textbox(label="Selected coordinates")
            #output_img = gr.Image(label = "Result", type = "numpy")

    #Create the next button
    with gr.Row():
        next_btn = gr.Button("Next ->")
    #print the picture            
    scan_canvas.select(fn=draw_circle, inputs=[scan_canvas], outputs=[scan_canvas, coords_output])
    #Get the next button working
    next_btn.click(
        fn=load_image, inputs=[], outputs=[scan_canvas]
    )

# Start the app
demo.launch(share=False)

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.
